# RL Teaching — Day 1
**Bandit · 2D Maze Q-Learning · Q-Table Heatmap**

> 📌 **Before you start:** Go to **File → Save a copy in Drive** to make your own editable copy.

You don't need to understand every line of code.
Focus on **what changes when you adjust the 🔧 parameters** and re-run the cell.

In [ ]:
# ── Setup (run this first) ──────────────────────────────────
!pip install matplotlib seaborn numpy --quiet
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print('✅ Setup complete!')

---
## Part 1 · Multi-Armed Bandit

Imagine a row of slot machines. Each machine has a *hidden* probability of giving a reward.
The agent uses **ε-greedy strategy**:
- With probability **ε** → try a random machine *(explore)*
- With probability **1 - ε** → pick the best-known machine *(exploit)*

### 🔧 Your task
Run the cell with **ε = 0.1**, then change it to **ε = 0.9** and run again.
What is different about the two reward curves?

In [ ]:
# ════════════════════════════════════════════
epsilon = 0.1   # 🔧 Try: 0.1 / 0.5 / 0.9
# ════════════════════════════════════════════

np.random.seed(42)
n_bandits, n_rounds = 5, 500
true_probs = [0.2, 0.5, 0.8, 0.4, 0.6]   # hidden from the agent

Q = np.zeros(n_bandits)
N = np.zeros(n_bandits)
rewards = []

for t in range(n_rounds):
    action = np.random.randint(n_bandits) if np.random.random() < epsilon else np.argmax(Q)
    reward = 1 if np.random.random() < true_probs[action] else 0
    N[action] += 1
    Q[action] += (reward - Q[action]) / N[action]
    rewards.append(reward)

smoothed = np.convolve(rewards, np.ones(30)/30, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(smoothed, linewidth=2, label=f'epsilon = {epsilon}')
plt.axhline(max(true_probs), color='gray', linestyle='--', label='Best possible reward')
plt.xlabel('Round'); plt.ylabel('Avg Reward (smoothed)')
plt.title(f'epsilon-greedy Bandit  |  epsilon = {epsilon}')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f"Agent best guess : Machine #{np.argmax(Q)} (Q = {max(Q):.2f})")
print(f"Actual best      : Machine #{np.argmax(true_probs)} (p = {max(true_probs):.2f})")

---
## Part 2 · 2D Maze — Q-Learning

The agent navigates a 5×5 grid maze from **S** (top-left) to **G** (bottom-right).
**#** cells are walls — the agent cannot enter them.

```
S  .  .  .  .
.  #  .  #  .
.  .  .  .  .
.  #  .  #  .
.  .  .  .  G
```

**Reward design** (same as Rein Room):
- Reach goal  →  **+1.0**
- Each step   →  **-0.01**  (encourages finding shorter paths)
- Hit a wall  →  **-0.05**  (stay in place)

Actions: 0 = Up, 1 = Down, 2 = Left, 3 = Right

### 🔧 Your task
Run with default settings. Then try changing **one parameter at a time**.
After training, look at the heatmap — can you trace a path from S to G?

In [ ]:
# ════════════════════════════════════════════
alpha      = 0.5    # 🔧 learning rate    — try: 0.1 / 0.5 / 0.9
gamma      = 0.95   # 🔧 discount factor  — try: 0.5 / 0.95 / 0.99
epsilon    = 0.2    # 🔧 exploration rate — try: 0.05 / 0.2 / 0.5
n_episodes = 1000
# ════════════════════════════════════════════

# ── Maze definition ─────────────────────────
ROWS, COLS = 5, 5
WALLS = {(1,1), (1,3), (3,1), (3,3)}
START, GOAL = (0,0), (4,4)
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]   # Up Down Left Right
ACT_NAMES = ['U','D','L','R']
N_STATES  = ROWS * COLS
N_ACTIONS = 4

def state_id(r, c): return r * COLS + c

def step(r, c, a):
    nr, nc = r + ACTIONS[a][0], c + ACTIONS[a][1]
    if nr < 0 or nr >= ROWS or nc < 0 or nc >= COLS or (nr,nc) in WALLS:
        return r, c, -0.05, False   # hit wall
    if (nr, nc) == GOAL:
        return nr, nc, 1.0, True    # reached goal
    return nr, nc, -0.01, False     # normal step

# ── Q-Learning ──────────────────────────────
Q = np.zeros((N_STATES, N_ACTIONS))
episode_rewards, episode_lengths = [], []

for _ in range(n_episodes):
    r, c = START
    total, steps, done = 0.0, 0, False
    while not done and steps < 200:
        s = state_id(r, c)
        a = np.random.randint(N_ACTIONS) if np.random.random() < epsilon else np.argmax(Q[s])
        nr, nc, reward, done = step(r, c, a)
        ns = state_id(nr, nc)
        Q[s, a] += alpha * (reward + gamma * np.max(Q[ns]) - Q[s, a])
        r, c, total, steps = nr, nc, total + reward, steps + 1
    episode_rewards.append(total)
    episode_lengths.append(steps)

window = 50
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(np.convolve(episode_rewards, np.ones(window)/window, mode='valid'), color='steelblue', lw=2)
ax1.set(title=f'Episode Reward | alpha={alpha} gamma={gamma} epsilon={epsilon}',
        xlabel='Episode', ylabel='Total Reward')
ax1.grid(alpha=0.3)
ax2.plot(np.convolve(episode_lengths, np.ones(window)/window, mode='valid'), color='coral', lw=2)
ax2.set(title='Episode Length (fewer steps = better path)', xlabel='Episode', ylabel='Steps')
ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
success = sum(1 for r in episode_rewards[-200:] if r > 0)
print(f"Success rate (last 200 episodes): {success/200:.1%}")

---
## Part 3 · Q-Table Heatmap

The Q-table stores a value for every (cell, direction) pair.
- **Arrow** → the direction the agent currently prefers in that cell
- **Color** → how confident the agent is (brighter = more certain)

This is the same visualization you see on the Rein Room platform.

### 🔧 Your task
1. Can you trace a path from **S** to **G** by following the arrows?
2. Change `cell_to_inspect` (row, col) to see Q-values for any specific cell.
3. Which area has the clearest arrows? Which area is still uncertain? Why?

In [ ]:
# ════════════════════════════════════════════
cell_to_inspect = (3, 4)   # 🔧 (row, col) — try cells along the path
# ════════════════════════════════════════════

best_act  = np.argmax(Q, axis=1).reshape(ROWS, COLS)
max_Q_val = np.max(Q, axis=1).reshape(ROWS, COLS)
arrows    = np.vectorize(lambda x: ACT_NAMES[x])(best_act)

# Mark walls, start, goal
display_label = np.full((ROWS, COLS), '', dtype=object)
display_label[START[0], START[1]] = 'S'
display_label[GOAL[0],  GOAL[1]]  = 'G'
for wr, wc in WALLS:
    display_label[wr, wc] = '#'
    max_Q_val[wr, wc] = np.nan

# Combine arrow + special label
annot = np.where(display_label != '', display_label, arrows)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(max_Q_val, annot=annot, fmt='', cmap='YlOrRd',
            linewidths=1, ax=ax1, cbar_kws={'label': 'Max Q-value (confidence)'})
ax1.set(title='Best Action per Cell  (arrow = preferred direction, color = confidence)',
        xlabel='Column', ylabel='Row')

# Bar chart for selected cell
r_i, c_i = cell_to_inspect
s_i = state_id(r_i, c_i)
q_vals = Q[s_i]
colors = ['#e74c3c','#2ecc71','#3498db','#f39c12']
bars = ax2.bar(ACT_NAMES, q_vals, color=colors)
ax2.set(title=f'Q-values for cell ({r_i},{c_i})',
        xlabel='Action', ylabel='Q-value')
ax2.grid(alpha=0.3, axis='y')
for bar, val in zip(bars, q_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.3f}',
             ha='center', va='bottom', fontsize=11)
plt.tight_layout(); plt.show()
print(f"Cell {cell_to_inspect}: preferred direction = {ACT_NAMES[np.argmax(q_vals)]}")